<a target="_blank" href="https://colab.research.google.com/github/genomicsxai/alphagenome-pytorch/blob/main/examples/notebooks/pytorch_forward_pass.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AlphaGenome PyTorch: Forward Pass Example

This notebook demonstrates how to run a forward pass using the PyTorch implementation of AlphaGenome and visualize the predictions.

**Requirements:**
- Model weights (`model.pth`)
- Track means (`track_means.pt`)
- The `alphagenome_pytorch` package

To convert JAX weights to PyTorch format:
```bash
python scripts/convert_weights.py /path/to/jax-checkpoint --output model.pth
python scripts/extract_track_means.py --output track_means.pt
```

## Setup

In [ ]:
# Uncomment to install in Colab
# %pip install /content/alphagenome-pytorch

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from alphagenome_pytorch import AlphaGenome

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## Configuration

Update these paths to point to your model weights and track means files.

In [ ]:
# Update these paths
MODEL_WEIGHTS_PATH = '../../model.pth'  # or your path to model weights
TRACK_MEANS_PATH = '../../track_means.pt'  # or your path to track means

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Load Model

In [ ]:
# Load track means for proper output scaling
track_means_dict = torch.load(TRACK_MEANS_PATH, weights_only=True)

# Initialize model
model = AlphaGenome(
    num_organisms=2,  # 0=human, 1=mouse
    track_means_dict=track_means_dict
)

# Load pretrained weights
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, weights_only=True), strict=False)
model.eval()
model.to(device)

print("Model loaded successfully!")

## Prepare Input Sequence

AlphaGenome expects one-hot encoded DNA sequences with shape `(batch, length, 4)` where channels represent A, C, G, T.

In [ ]:
# Generate a random DNA sequence for demonstration
# In practice, you would load real genomic sequences
sequence_length = 1048576  # 1M bp (2^20)

# Random sequence: 0=A, 1=C, 2=G, 3=T
np.random.seed(42)
random_seq = np.random.randint(0, 4, size=(1, sequence_length))

# One-hot encode
dna_one_hot = np.eye(4)[random_seq]  # (1, length, 4)
dna_one_hot = torch.tensor(dna_one_hot, dtype=torch.float32).to(device)

# Organism index: 0 = human, 1 = mouse
organism_index = torch.tensor([0], dtype=torch.long).to(device)

print(f"Input shape: {dna_one_hot.shape}")
print(f"Organism: Human (index={organism_index.item()})")

## Run Forward Pass

In [ ]:
%%time

with torch.no_grad():
    outputs = model(dna_one_hot, organism_index)

print(f"\nAvailable output types: {list(outputs.keys())}")

## Explore Outputs

Each output type has predictions at different resolutions (1bp and/or 128bp).

In [ ]:
# Print output shapes
for name, out in outputs.items():
    if isinstance(out, dict):
        for res, tensor in out.items():
            print(f"{name}[{res}]: {tensor.shape}")
    elif isinstance(out, torch.Tensor):
        print(f"{name}: {out.shape}")
    else:
        print(f"{name}: {type(out)}")

## Visualize Predictions

Let's plot some example track predictions.

In [ ]:
def plot_tracks(outputs, track_names, resolution=128, window_size=1024, start=0, track_indices=None):
    """
    Plot genomic track predictions.
    
    Args:
        outputs: Model output dictionary
        track_names: List of track names to plot (e.g., ['atac', 'dnase'])
        resolution: Resolution to plot (1 or 128)
        window_size: Number of bins to show
        start: Starting bin position
        track_indices: Optional dict mapping track_name to indices to plot
    """
    n_tracks = len(track_names)
    fig, axes = plt.subplots(n_tracks, 1, figsize=(14, 3 * n_tracks), sharex=True)
    
    if n_tracks == 1:
        axes = [axes]
    
    end = start + window_size
    x = np.arange(start, end) * resolution  # Convert to bp
    
    for ax, name in zip(axes, track_names):
        if name not in outputs or resolution not in outputs[name]:
            ax.set_title(f"{name} (not available at {resolution}bp)")
            continue
            
        data = outputs[name][resolution][0, start:end, :].cpu().numpy()
        
        # Select which track indices to plot
        if track_indices and name in track_indices:
            idx = track_indices[name]
        else:
            # Plot mean across all tracks
            idx = None
        
        if idx is not None:
            for i in idx:
                ax.plot(x, data[:, i], alpha=0.7, label=f"Track {i}")
            ax.legend(loc='upper right', fontsize=8)
        else:
            # Plot mean and std
            mean = data.mean(axis=1)
            std = data.std(axis=1)
            ax.plot(x, mean, color='blue', alpha=0.8)
            ax.fill_between(x, mean - std, mean + std, alpha=0.3, color='blue')
        
        ax.set_ylabel(name.upper())
        ax.set_title(f"{name.upper()} predictions at {resolution}bp resolution")
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel('Position (bp)')
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot ATAC, DNase, and CAGE predictions at 128bp resolution
plot_tracks(
    outputs, 
    track_names=['atac', 'dnase', 'cage'],
    resolution=128,
    window_size=512,
    start=1000
)

In [ ]:
# Plot specific tracks at 1bp resolution
plot_tracks(
    outputs,
    track_names=['atac'],
    resolution=1,
    window_size=2048,
    start=500000,
    track_indices={'atac': [0, 1, 2]}  # Plot first 3 tracks
)

## RNA-seq Predictions

In [ ]:
# RNA-seq has squashing applied (log1p-like transformation)
rna_seq_128bp = outputs['rna_seq'][128][0].cpu().numpy()

plt.figure(figsize=(14, 4))
plt.imshow(
    rna_seq_128bp.T[:50, 1000:1500],  # First 50 tracks, window of 500 bins
    aspect='auto',
    cmap='viridis'
)
plt.colorbar(label='Predicted signal')
plt.xlabel('Position (128bp bins)')
plt.ylabel('Track index')
plt.title('RNA-seq predictions (first 50 tracks)')
plt.tight_layout()
plt.show()

## Contact Map Predictions

In [ ]:
# Contact/Hi-C predictions are 2D
if 'pair_activations' in outputs:
    contact_maps = outputs['pair_activations']
    
    # The contact map output has shape (B, S, S, num_tracks)
    # Plot the first track
    if isinstance(contact_maps, torch.Tensor):
        cm = contact_maps[0, :, :, 0].cpu().numpy()
    else:
        cm = contact_maps[128][0, :, :, 0].cpu().numpy()
    
    plt.figure(figsize=(8, 8))
    plt.imshow(cm, cmap='RdBu_r', vmin=-2, vmax=2)
    plt.colorbar(label='Contact score')
    plt.title('Predicted contact map (track 0)')
    plt.xlabel('Position')
    plt.ylabel('Position')
    plt.tight_layout()
    plt.show()
else:
    print("Contact map predictions not available")

## Summary

This notebook demonstrated:
1. Loading the AlphaGenome PyTorch model with track means
2. Preparing one-hot encoded DNA input
3. Running a forward pass
4. Visualizing predictions for various track types

For more advanced usage including variant effect prediction, see the other example notebooks.